# Análise de Agrupamento: K-means e PCA

**Projeto:** Análise de Dados de Compliance Público (TCC MBA)  
**Autor:** Enok  
**Última Atualização:** 2026-02-10

---

## Objetivo

Este notebook realiza análise de agrupamento em municípios brasileiros utilizando indicadores socioeconômicos dos Censos de 2010 e 2022. A análise inclui:

1. **Estatísticas Descritivas** - Resumo do dataset consolidado
2. **PCA (Análise de Componentes Principais)** - Redução de dimensionalidade para identificar componentes de variância
3. **Agrupamento K-means** - Agrupar municípios baseado em:
   - População
   - Taxas de alfabetização
   - Renda
   - Indicadores sociais de domicílios
4. **Perfil dos Clusters** - Caracterização de cada grupo

---

## Fonte de Dados

- **Dataset:** `gold/consolidated_clustering/data.parquet`
- **Granularidade:** Um registro por município (cidade)
- **Características:**
  - Sem municípios duplicados
  - Sem valores ausentes nas features de agrupamento
  - Features pré-normalizadas (padronização z-score)
  - Dados de ambos os anos censitários (2010 e 2022)

## Passo a passo (estilo aula)

1. Pacotes e configuração do ambiente
2. Reprodutibilidade
3. Carregamento dos dados
4. Blocos de análise
5. Resumo e interpretação


## 1. Configuração e Carregamento de Dados

# Pacotes


In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Adicionar raiz do projeto ao path
project_root = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Bibliotecas principais
import numpy as np
import pandas as pd

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

# AWS
import boto3
import tempfile

# Configurar estilo de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("Bibliotecas carregadas com sucesso!")


# Reprodutibilidade


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Semente de reprodutibilidade fixada em {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


In [ ]:
# Configuração
BUCKET_NAME = S3_BUCKET_NAME
DATA_KEY = "gold/consolidated_clustering/data.parquet"

# Carregar dados do S3
def load_from_s3(bucket: str, key: str) -> pd.DataFrame:
    """Carregar arquivo parquet do S3."""
    aws_profile = os.getenv("AWS_PROFILE") or AWS_PROFILE
    session = boto3.Session(profile_name=aws_profile)
    s3 = session.client('s3')
    with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
        s3.download_file(bucket, key, tmp.name)
        return pd.read_parquet(tmp.name)

# Carregar o dataset consolidado
df = load_from_s3(BUCKET_NAME, DATA_KEY)
print(f"Carregados {len(df):,} municípios")
print(f"Colunas: {len(df.columns)}")


In [ ]:
# Exibir primeiras linhas
df.head()


In [ ]:
# Verificar qualidade dos dados
print("=" * 60)
print("VERIFICAÇÃO DE QUALIDADE DOS DADOS")
print("=" * 60)
print(f"\nTotal de municípios: {len(df):,}")
print(f"Municípios únicos: {df['municipality_code'].nunique():,}")
print(f"Municípios duplicados: {len(df) - df['municipality_code'].nunique()}")
print(f"\nValores ausentes por coluna:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "Sem valores ausentes!")


## 2. Estatísticas Descritivas

In [ ]:
# Definir colunas de features brutas (não normalizadas)
raw_features = [
    'population_2010', 'population_2022', 'population_change_pct',
    'literacy_rate_2010', 'literacy_rate_2022', 'literacy_change_pp',
    'avg_income_real_2010_2022_brl', 'avg_income_real_2022_2022_brl', 'income_change_real_pct',
    'households_2010', 'households_2022', 'households_change_pct'
]

# Estatísticas descritivas
print("=" * 80)
print("ESTATÍSTICAS DESCRITIVAS - Features Brutas")
print("=" * 80)
df[raw_features].describe().round(2).T


In [ ]:
# Distribuição por região
print("\n" + "=" * 60)
print("DISTRIBUIÇÃO POR REGIÃO")
print("=" * 60)

region_stats = df.groupby('region_name').agg({
    'municipality_code': 'count',
    'population_2022': ['sum', 'mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_real_2022_2022_brl': 'mean'
}).round(2)

region_stats.columns = ['N_Municipios', 'Pop_Total', 'Pop_Media', 'Pop_Mediana', 
                        'Alfabetizacao_Media', 'Renda_Media']
region_stats = region_stats.sort_values('N_Municipios', ascending=False)
region_stats


In [ ]:
# Distribuição por estado
print("\n" + "=" * 60)
print("TOP 10 ESTADOS POR NÚMERO DE MUNICÍPIOS")
print("=" * 60)

state_counts = df.groupby(['state_name', 'region_name']).size().reset_index(name='n_municipios')
state_counts = state_counts.sort_values('n_municipios', ascending=False).head(10)
state_counts


In [ ]:
# Visualizar distribuições das features principais
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# População 2022 (escala log)
ax = axes[0, 0]
ax.hist(np.log10(df['population_2022']), bins=50, edgecolor='white', alpha=0.7)
ax.set_xlabel('Log10(População 2022)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de População (2022)')

# Taxa de Alfabetização 2022
ax = axes[0, 1]
ax.hist(df['literacy_rate_2022'], bins=50, edgecolor='white', alpha=0.7, color='green')
ax.set_xlabel('Taxa de Alfabetização (%)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição da Taxa de Alfabetização (2022)')

# Renda 2022
ax = axes[0, 2]
ax.hist(df['avg_income_real_2022_2022_brl'], bins=50, edgecolor='white', alpha=0.7, color='orange')
ax.set_xlabel('Renda Média (R$)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição de Renda (2022)')

# Variação da População
ax = axes[1, 0]
ax.hist(df['population_change_pct'], bins=50, edgecolor='white', alpha=0.7, color='purple')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da População (%)')
ax.set_ylabel('Frequência')
ax.set_title('Variação Populacional (2010-2022)')

# Variação da Alfabetização
ax = axes[1, 1]
ax.hist(df['literacy_change_pp'], bins=50, edgecolor='white', alpha=0.7, color='teal')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da Alfabetização (pp)')
ax.set_ylabel('Frequência')
ax.set_title('Variação da Alfabetização (2010-2022)')

# Variação da Renda
ax = axes[1, 2]
ax.hist(df['income_change_real_pct'], bins=50, edgecolor='white', alpha=0.7, color='brown')
ax.axvline(x=0, color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Variação da Renda (%)')
ax.set_ylabel('Frequência')
ax.set_title('Variação da Renda (2010-2022)')

plt.tight_layout()
plt.show()


In [ ]:
# Matriz de correlação
plt.figure(figsize=(14, 10))
corr_matrix = df[raw_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', square=True, linewidths=0.5)
plt.title('Matriz de Correlação - Features Socioeconômicas', fontsize=14)
plt.tight_layout()
plt.show()


## 3. PCA - Análise de Componentes Principais

O PCA nos ajudará a:
1. Reduzir dimensionalidade preservando a variância
2. Identificar quais features mais contribuem para a variância
3. Visualizar municípios em espaço 2D/3D
4. Potencialmente usar menos componentes para agrupamento

In [ ]:
# Usar features normalizadas para PCA
norm_features = [f'{col}_norm' for col in raw_features]

# Extrair dados normalizados
X_norm = df[norm_features].values

print(f"Formato da matriz de features: {X_norm.shape}")
print(f"Número de features: {len(norm_features)}")


In [ ]:
# Executar PCA com todos os componentes
pca_full = PCA()
pca_full.fit(X_norm)

# Variância explicada
explained_var = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# Exibir variância explicada
print("=" * 60)
print("PCA - VARIÂNCIA EXPLICADA")
print("=" * 60)
pca_df = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(len(explained_var))],
    'Variancia_Explicada': explained_var * 100,
    'Variancia_Acumulada': cumulative_var * 100
})
print(pca_df.round(2).to_string(index=False))


In [ ]:
# Gráfico Scree e variância acumulada
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico Scree
ax = axes[0]
components = range(1, len(explained_var) + 1)
ax.bar(components, explained_var * 100, alpha=0.7, label='Individual')
ax.plot(components, cumulative_var * 100, 'ro-', label='Acumulada')
ax.axhline(y=80, color='green', linestyle='--', label='Limiar 80%')
ax.set_xlabel('Componente Principal')
ax.set_ylabel('Variância Explicada (%)')
ax.set_title('Gráfico Scree do PCA')
ax.legend()
ax.set_xticks(components)

# Variância acumulada
ax = axes[1]
ax.plot(components, cumulative_var * 100, 'b-o', linewidth=2, markersize=8)
ax.axhline(y=80, color='green', linestyle='--', label='Limiar 80%')
ax.axhline(y=90, color='orange', linestyle='--', label='Limiar 90%')
ax.axhline(y=95, color='red', linestyle='--', label='Limiar 95%')
ax.fill_between(components, cumulative_var * 100, alpha=0.3)
ax.set_xlabel('Número de Componentes')
ax.set_ylabel('Variância Acumulada Explicada (%)')
ax.set_title('Variância Acumulada Explicada')
ax.legend()
ax.set_xticks(components)

plt.tight_layout()
plt.show()

# Determinar componentes ótimos
n_components_80 = np.argmax(cumulative_var >= 0.80) + 1
n_components_90 = np.argmax(cumulative_var >= 0.90) + 1
print(f"\nComponentes necessários para 80% da variância: {n_components_80}")
print(f"Componentes necessários para 90% da variância: {n_components_90}")


In [ ]:
# Cargas do PCA (contribuições das features para cada componente)
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f'PC{i+1}' for i in range(len(explained_var))],
    index=[col.replace('_norm', '') for col in norm_features]
)

print("=" * 60)
print("CARGAS DO PCA (Contribuições das Features)")
print("=" * 60)
print(loadings[['PC1', 'PC2', 'PC3', 'PC4']].round(3))


In [ ]:
# Visualizar heatmap das cargas
plt.figure(figsize=(12, 8))
sns.heatmap(loadings[['PC1', 'PC2', 'PC3', 'PC4']], annot=True, cmap='RdBu_r', 
            center=0, fmt='.2f', linewidths=0.5)
plt.title('Cargas do PCA - Contribuições das Features para Componentes Principais', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Transformar dados para componentes principais
pca_3 = PCA(n_components=3)
X_pca = pca_3.fit_transform(X_norm)

# Adicionar componentes PCA ao dataframe
df['PC1'] = X_pca[:, 0]
df['PC2'] = X_pca[:, 1]
df['PC3'] = X_pca[:, 2]

print(f"Transformação PCA concluída.")
print(f"Variância explicada por 3 componentes: {pca_3.explained_variance_ratio_.sum()*100:.1f}%")


In [ ]:
# Visualização 2D do PCA por região
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='region_name',
    hover_data=['municipality_name', 'state_name', 'population_2022', 'avg_income_real_2022_2022_brl'],
    title='PCA: Municípios no Espaço 2D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)'}
)
fig.update_layout(height=600)
fig.show()


In [ ]:
# Visualização 3D do PCA
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='region_name',
    hover_data=['municipality_name', 'state_name'],
    title='PCA: Municípios no Espaço 3D (por Região)',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'PC3': f'PC3 ({pca_3.explained_variance_ratio_[2]*100:.1f}%)'}
)
fig.update_layout(height=700)
fig.show()


## 4. Agrupamento K-means

Utilizaremos K-means para agrupar municípios baseado em indicadores socioeconômicos:
- População
- Taxas de alfabetização
- Renda
- Indicadores de domicílios

### 4.1 Determinar Número Ótimo de Clusters

In [ ]:
# Selecionar features para agrupamento (usando features normalizadas)
clustering_features = [
    'population_2022_norm',
    'literacy_rate_2022_norm',
    'avg_income_real_2022_2022_brl_norm',
    'households_2022_norm',
    'population_change_pct_norm',
    'literacy_change_pp_norm',
    'income_change_real_pct_norm',
    'households_change_pct_norm'
]

X_cluster = df[clustering_features].values
print(f"Matriz de features para agrupamento: {X_cluster.shape}")
print(f"Features utilizadas: {clustering_features}")


In [ ]:
# Método do Cotovelo e Análise de Silhueta
k_range = range(2, 11)
inertias = []
silhouettes = []

print("Avaliando valores de K...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_cluster)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_cluster, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"  K={k}: Inércia={kmeans.inertia_:.0f}, Silhueta={sil_score:.4f}")


In [ ]:
# Plotar Cotovelo e Silhueta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico do Cotovelo
ax = axes[0]
ax.plot(list(k_range), inertias, 'b-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Inércia (Soma dos quadrados intra-cluster)')
ax.set_title('Método do Cotovelo')
ax.set_xticks(list(k_range))

# Gráfico da Silhueta
ax = axes[1]
ax.plot(list(k_range), silhouettes, 'g-o', linewidth=2, markersize=8)
ax.set_xlabel('Número de Clusters (K)')
ax.set_ylabel('Coeficiente de Silhueta')
ax.set_title('Análise de Silhueta')
ax.set_xticks(list(k_range))

# Destacar melhor silhueta
best_k = list(k_range)[np.argmax(silhouettes)]
best_sil = max(silhouettes)
ax.axvline(x=best_k, color='red', linestyle='--', label=f'Melhor K={best_k}')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nMelhor K baseado no Coeficiente de Silhueta: {best_k} (score={best_sil:.4f})")


### 4.2 Aplicar K-means com K Ótimo

In [ ]:
# Usar K baseado na análise (ajustar conforme necessário)
OPTIMAL_K = best_k
print(f"Usando K = {OPTIMAL_K} clusters")

# Ajustar modelo K-means final
kmeans_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
df['cluster'] = kmeans_final.fit_predict(X_cluster)

# Distribuição dos clusters
print("\nDistribuição dos Clusters:")
print(df['cluster'].value_counts().sort_index())


In [ ]:
# Visualizar clusters no espaço PCA (2D)
fig = px.scatter(
    df, x='PC1', y='PC2',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name', 'region_name', 'population_2022', 'avg_income_real_2022_2022_brl'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA',
    labels={'PC1': f'PC1 ({pca_3.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca_3.explained_variance_ratio_[1]*100:.1f}%)',
            'cluster': 'Cluster'}
)
fig.update_traces(marker=dict(size=5))
fig.update_layout(height=600)
fig.show()


In [ ]:
# Visualizar clusters no espaço PCA 3D
fig = px.scatter_3d(
    df, x='PC1', y='PC2', z='PC3',
    color='cluster',
    color_continuous_scale='viridis',
    hover_data=['municipality_name', 'state_name'],
    title=f'Clusters K-means (K={OPTIMAL_K}) no Espaço PCA 3D'
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(height=700)
fig.show()


### 4.3 Perfil dos Clusters

In [ ]:
# Estatísticas dos clusters
print("=" * 80)
print("PERFIS DOS CLUSTERS - Valores Médios")
print("=" * 80)

cluster_stats = df.groupby('cluster').agg({
    'municipality_code': 'count',
    'population_2022': ['mean', 'median'],
    'literacy_rate_2022': 'mean',
    'avg_income_real_2022_2022_brl': 'mean',
    'households_2022': 'mean',
    'population_change_pct': 'mean',
    'literacy_change_pp': 'mean',
    'income_change_real_pct': 'mean'
}).round(2)

cluster_stats.columns = ['N_Municipios', 'Pop_Media', 'Pop_Mediana', 'Alfabetizacao_Media',
                         'Renda_Media', 'Domicilios_Media', 'Var_Pop', 'Var_Alfab', 'Var_Renda']
cluster_stats


In [ ]:
# Composição dos clusters por região
print("\n" + "=" * 60)
print("COMPOSIÇÃO DOS CLUSTERS POR REGIÃO")
print("=" * 60)

region_cluster = pd.crosstab(df['cluster'], df['region_name'], margins=True)
print(region_cluster)


In [ ]:
# Visualizar composição dos clusters por região
fig = px.histogram(
    df, x='cluster', color='region_name',
    barmode='stack',
    title='Composição dos Clusters por Região',
    labels={'cluster': 'Cluster', 'count': 'Número de Municípios'}
)
fig.update_layout(height=500)
fig.show()


In [ ]:
# Box plots para cada feature por cluster
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

features_to_plot = [
    ('population_2022', 'População 2022', True),
    ('literacy_rate_2022', 'Taxa Alfabetização 2022 (%)', False),
    ('avg_income_real_2022_2022_brl', 'Renda Média 2022 (R$)', False),
    ('households_2022', 'Domicílios 2022', True),
    ('population_change_pct', 'Var. População (%)', False),
    ('literacy_change_pp', 'Var. Alfabetização (pp)', False),
    ('income_change_real_pct', 'Var. Renda (%)', False),
    ('households_change_pct', 'Var. Domicílios (%)', False)
]

for ax, (col, title, use_log) in zip(axes, features_to_plot):
    data = np.log10(df[col]) if use_log else df[col]
    ylabel = f'Log10({col})' if use_log else col
    df.boxplot(column=col if not use_log else None, by='cluster', ax=ax)
    if use_log:
        for i, cluster in enumerate(sorted(df['cluster'].unique())):
            cluster_data = np.log10(df[df['cluster'] == cluster][col])
            ax.boxplot(cluster_data, positions=[i+1])
    ax.set_title(title)
    ax.set_xlabel('Cluster')

plt.suptitle('Distribuições das Features por Cluster', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Gráfico radar para perfis dos clusters
# Normalizar médias dos clusters para comparação
cluster_means = df.groupby('cluster')[raw_features].mean()
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min())

# Selecionar features principais para o radar
radar_features = ['population_2022', 'literacy_rate_2022', 'avg_income_real_2022_2022_brl', 
                  'households_2022', 'population_change_pct', 'income_change_real_pct']

fig = go.Figure()

for cluster in sorted(df['cluster'].unique()):
    values = cluster_means_norm.loc[cluster, radar_features].values.tolist()
    values.append(values[0])  # Fechar o radar
    
    fig.add_trace(go.Scatterpolar(
        r=values,
        theta=radar_features + [radar_features[0]],
        fill='toself',
        name=f'Cluster {cluster}'
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1])
    ),
    title='Perfis dos Clusters (Normalizados)',
    height=600
)
fig.show()


### 4.4 Interpretação dos Clusters

In [ ]:
# Gerar interpretações dos clusters baseado nas estatísticas
print("=" * 80)
print("INTERPRETAÇÃO DOS CLUSTERS")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    cluster_data = df[df['cluster'] == cluster]
    n_muni = len(cluster_data)
    
    print(f"\n--- CLUSTER {cluster} ({n_muni} municípios, {100*n_muni/len(df):.1f}%) ---")
    
    # População
    pop_mean = cluster_data['population_2022'].mean()
    pop_median = cluster_data['population_2022'].median()
    pop_size = "Grande" if pop_mean > df['population_2022'].mean() else "Pequena"
    print(f"  População: {pop_size} (média={pop_mean:,.0f}, mediana={pop_median:,.0f})")
    
    # Alfabetização
    lit_mean = cluster_data['literacy_rate_2022'].mean()
    lit_level = "Alta" if lit_mean > df['literacy_rate_2022'].mean() else "Baixa"
    print(f"  Alfabetização: {lit_level} ({lit_mean:.1f}%)")
    
    # Renda
    inc_mean = cluster_data['avg_income_real_2022_2022_brl'].mean()
    inc_level = "Alta" if inc_mean > df['avg_income_real_2022_2022_brl'].mean() else "Baixa"
    print(f"  Renda: {inc_level} (R$ {inc_mean:,.2f})")
    
    # Crescimento
    pop_change = cluster_data['population_change_pct'].mean()
    growth = "Crescendo" if pop_change > 0 else "Declinando"
    print(f"  Tendência Populacional: {growth} ({pop_change:+.1f}%)")
    
    # Principais regiões
    top_regions = cluster_data['region_name'].value_counts().head(2)
    print(f"  Principais Regiões: {', '.join(top_regions.index)}")


In [ ]:
# Exemplos de municípios de cada cluster
print("\n" + "=" * 80)
print("EXEMPLOS DE MUNICÍPIOS DE CADA CLUSTER")
print("=" * 80)

for cluster in sorted(df['cluster'].unique()):
    print(f"\n--- Cluster {cluster} ---")
    examples = df[df['cluster'] == cluster].nlargest(5, 'population_2022')[
        ['municipality_name', 'state_name', 'population_2022', 'avg_income_real_2022_2022_brl', 'literacy_rate_2022']
    ]
    print(examples.to_string(index=False))


## 5. Resumo e Conclusões

### 5.1 Dataset
- **5.565 municípios** analisados (99,9% de 5.570 no total), com 12 features socioeconômicas cobrindo população, alfabetização, renda (ajustada pela inflação) e domicílios para 2010 e 2022.
- Sem valores ausentes ou duplicatas no dataset final de clustering.

### 5.2 Resultados do PCA
- **3 componentes principais** explicam **84,4%** da variância total; 4 componentes alcançam 92,5%.
- **PC1 (39,2%)**: Desenvolvimento socioeconômico geral — carrega em população, renda, alfabetização e domicílios.
- **PC2 (28,8%)**: Escala urbana vs desenvolvimento — separa grandes centros populacionais de municípios com alta alfabetização/renda.
- **PC3 (16,5%)**: Trajetória de crescimento — carrega em variação populacional e de domicílios entre 2010–2022.

### 5.3 Clustering K-Means
- **K ótimo = 4** clusters, selecionado pelo silhouette score (0,288).
- **Cluster 0** (821 municípios, 14,8%): Grandes, alta renda, alta alfabetização, população crescente — principais centros urbanos (Brasília, Fortaleza, Salvador, Belo Horizonte). Concentrados no Sudeste e Sul.
- **Cluster 1** (2.158 municípios, 38,8%): Pequenos, baixa renda, baixa alfabetização, população estagnada/declinante — predominantemente Nordeste e Norte. Representa o gap de desenvolvimento.
- **Cluster 2** (2.584 municípios, 46,4%): Pequenos a médios, alta alfabetização, alta renda, mas população em declínio — principalmente municípios do interior do Sudeste e Sul.
- **Cluster 3** (2 municípios, 0,04%): Megacidades (São Paulo e Rio de Janeiro) — outliers extremos de população formando seu próprio cluster.

### 5.4 Insights Principais
- Os municípios brasileiros mostram uma clara **estrutura dual**: um cluster desenvolvido Sul/Sudeste (Clusters 0, 2, 3) vs um cluster menos desenvolvido Norte/Nordeste (Cluster 1).
- **Renda e alfabetização** são os principais diferenciadores entre clusters, confirmando a divisão socioeconômica visível no PCA.
- **Declínio populacional** afeta municípios tanto desenvolvidos (Cluster 2) quanto menos desenvolvidos (Cluster 1), mas por razões diferentes — êxodo rural vs estagnação econômica.
- O **silhouette score moderado** (0,288) indica fronteiras sobrepostas entre clusters, consistente com um gradiente socioeconômico contínuo em vez de grupos nitidamente separados.

### 5.5 Limitações
- O clustering utiliza apenas indicadores derivados do censo — adicionar dados de sanções, fiscais ou de governança poderia revelar agrupamentos mais nuançados.
- K-means assume clusters esféricos; métodos baseados em densidade (DBSCAN, HDBSCAN) podem capturar melhor as distribuições assimétricas.
- O Cluster 3 com 2 municípios é um artefato de outliers extremos de população, não um grupo analítico significativo.
- Colunas de renda ajustada pela inflação usam deflação pelo IPCA para BRL de 2022 — resultados dependem da série de deflator utilizada.


In [ ]:
print("=" * 80)
print("RESUMO DA ANÁLISE")
print("=" * 80)

print(f"""
DATASET:
  - Total de municípios analisados: {len(df):,}
  - Features utilizadas: {len(raw_features)}
  - Sem valores ausentes ou duplicatas

RESULTADOS DO PCA:
  - Componentes necessários para 80% da variância: {n_components_80}
  - Componentes necessários para 90% da variância: {n_components_90}
  - Primeiros 3 componentes explicam: {pca_3.explained_variance_ratio_.sum()*100:.1f}% da variância

AGRUPAMENTO K-MEANS:
  - K ótimo: {OPTIMAL_K} clusters
  - Coeficiente de Silhueta: {silhouette_score(X_cluster, df['cluster']):.4f}
  - Tamanhos dos clusters: {dict(df['cluster'].value_counts().sort_index())}

PRINCIPAIS ACHADOS:
  - Municípios podem ser agrupados baseado em características socioeconômicas
  - Tamanho da população e renda são fatores diferenciadores importantes
  - Padrões regionais são visíveis na composição dos clusters
  - Trajetórias de crescimento (2010-2022) variam significativamente entre clusters
""")


In [ ]:
# Salvar dados com clusters
output_columns = ['municipality_code', 'municipality_name', 'state_code', 'state_name',
                  'region_code', 'region_name', 'cluster', 'PC1', 'PC2', 'PC3'] + raw_features

df_output = df[output_columns].copy()
print(f"Dataset de saída pronto com {len(df_output)} linhas e {len(output_columns)} colunas")
df_output.head()


In [ ]:
# Salvar no S3 (opcional)
# output_key = 'gold/clustered_municipalities/data.parquet'
# with tempfile.NamedTemporaryFile(suffix='.parquet') as tmp:
#     df_output.to_parquet(tmp.name, index=False)
#     aws_profile = os.getenv("AWS_PROFILE") or AWS_PROFILE
#     session = boto3.Session(profile_name=aws_profile)
#     s3 = session.client('s3')
#     s3.upload_file(tmp.name, BUCKET_NAME, output_key)
#     print(f"Salvo em s3://{BUCKET_NAME}/{output_key}")


---

## Fim da Análise

Este notebook demonstrou:
1. Carregamento e validação do dataset consolidado de municípios
2. Estatísticas descritivas e distribuições de features
3. PCA para redução de dimensionalidade (identificando componentes de variância principais)
4. Agrupamento K-means para agrupar municípios por características socioeconômicas
5. Perfil e interpretação dos clusters

**Próximos Passos:**
- Usar rótulos de clusters para análise estratificada
- Investigar padrões de compliance dentro de cada cluster
- Comparar características dos clusters com dados de sanções

## 6. Síntese Entre Notebooks

Esta análise de clustering é o quarto e último notebook analítico do pipeline da tese. Abaixo está um resumo de como os achados dos quatro notebooks se conectam.

### Cadeia de Evidências

| Notebook | Método | Achado Principal |
|----------|--------|------------------|
| **NB01 — EDA** | Estatística descritiva | Renda é o correlato mais forte de sanções por 100 mil (r = 0,74). Distrito Federal é outlier estrutural. Distribuição fortemente assimétrica à direita. |
| **NB02 — Estatística** | Regressão OLS | R² = 0,835 — renda + dummies regionais explicam 83,5% da variância das sanções. Norte e Nordeste têm sanções acima do esperado após controle pela renda. Alfabetização perde significância quando renda está presente. |
| **NB03 — Machine Learning** | ElasticNet, RF, Logística | Confirma renda como preditor dominante em todos os métodos. Classificação fraca com n = 27 (apenas exploratória). |
| **NB04 — Clustering** | PCA + K-means | Brasil tem estrutura municipal dual: Sul/Sudeste desenvolvido vs Norte/Nordeste menos desenvolvido. 3 PCs explicam 84,4% da variância. K = 4 clusters (silhouette = 0,288). |

### Argumento Convergente da Tese

As sanções de compliance público no Brasil **não são distribuídas aleatoriamente** — estão fortemente associadas a indicadores de desenvolvimento socioeconômico, particularmente renda, e exibem padrões regionais persistentes que sobrevivem aos controles estatísticos.

A explicação mais parcimoniosa é a **capacidade institucional de detecção**: jurisdições com maior renda possuem infraestrutura de auditoria mais forte, produzindo mais sanções registradas independentemente dos níveis reais de irregularidade. Norte e Nordeste mostram sanções acima do esperado após controle pela renda, sugerindo fatores de governança além do desenvolvimento socioeconômico.

O clustering em nível municipal confirma que a estrutura dual socioeconômica do Brasil (Sul/Sudeste desenvolvido vs Norte/Nordeste menos desenvolvido) molda tanto a geração de transferências públicas quanto a capacidade institucional de monitorá-las.

### Limitações

- **n = 27** em nível estadual limita o poder estatístico para modelos de regressão e ML
- **Desenho transversal** — causalidade não pode ser estabelecida
- **Viés de detecção** — sanções medem violações registradas, não irregularidades reais
- **Distrito Federal** — outlier estrutural que influencia todos os modelos
- **Deflação da renda** — resultados dependem da série de deflator IPCA para BRL de 2022

*Conclusão completa da tese: ver `docs/thesis_conclusion.pt-BR.md`*


## 7. Mapas do Brasil (achados em estilo QGIS no notebook)

Esta seção renderiza os achados finais da tese em mapas do Brasil sob duas perspectivas:

1. **Por estado**: intensidade de sanções por 100 mil, enriquecida com composição de clusters municipais.
2. **Por principais cidades**: municípios mais populosos, coloridos por cluster e anotados com métricas de compliance.

Os ativos de mapa são gerados a partir dos datasets Gold atuais e dos limites oficiais do IBGE.
        


In [ ]:
# Carregar datasets adicionais para os mapas finais de achados
STATE_FINDINGS_KEY = "gold/analysis_compliance/data.parquet"
CITY_FINDINGS_KEY = "gold/analysis_compliance_municipality/data.parquet"

from src.analysis.presentation_assets import build_state_final_findings

df_state = load_from_s3(BUCKET_NAME, STATE_FINDINGS_KEY)
df_city = load_from_s3(BUCKET_NAME, CITY_FINDINGS_KEY)

cluster_assignments = df[["municipality_code", "cluster"]].copy()
cluster_assignments["municipality_code"] = cluster_assignments["municipality_code"].astype(str).str.zfill(7)

df_state["state_code"] = df_state["state_code"].astype(str).str.zfill(2)
df_city["municipality_code"] = df_city["municipality_code"].astype(str).str.zfill(7)
df_city["state_code"] = df_city["state_code"].astype(str).str.zfill(2)

state_findings, cluster_state_mix = build_state_final_findings(
    state_df=df_state,
    city_df=df_city,
    cluster_assignments=cluster_assignments,
)

city_with_cluster = df_city.merge(cluster_assignments, on="municipality_code", how="left")

print(f"Linhas de achados por estado: {len(state_findings)}")
print(f"Linhas de achados por município: {len(city_with_cluster):,}")
state_findings.head()
        


In [ ]:
# Construir / carregar GeoJSON estadual a partir dos limites oficiais do IBGE
from pathlib import Path
import zipfile
import tempfile

try:
    import shapefile  # pyshp
except ImportError as exc:
    raise ImportError(
        "pyshp é necessário para renderização de mapas. Instale com: pip install pyshp>=2.3.1"
    ) from exc

from src.analysis.presentation_assets import download_ibge_boundary_zip, build_state_geojson_from_shapefile

map_assets_dir = Path("..") / "docs" / "thesis_presentation_assets" / "qgis"
map_assets_dir.mkdir(parents=True, exist_ok=True)

state_zip_path = map_assets_dir / "BR_UF_2022.zip"
if not state_zip_path.exists():
    state_zip_path = download_ibge_boundary_zip("BR_UF_2022.zip", output_dir=map_assets_dir)

state_geojson_path = map_assets_dir / "brazil_states_final_findings.geojson"
build_state_geojson_from_shapefile(
    state_zip_path=state_zip_path,
    state_findings=state_findings,
    output_geojson_path=state_geojson_path,
)

with open(state_geojson_path, "r", encoding="utf-8") as _f:
    state_geojson = _json.load(_f)

print(f"GeoJSON estadual pronto: {state_geojson_path}")
        


In [ ]:
# Mapa 1: Brasil por estado (sanções por 100 mil + contexto de clusters)
fig_state = px.choropleth_mapbox(
    state_findings,
    geojson=state_geojson,
    locations="state_code",
    featureidkey="properties.state_code",
    color="sanctions_per_100k_state",
    hover_name="state_name",
    hover_data={
        "region_name": True,
        "dominant_cluster": True,
        "dominant_cluster_share_pct": ":.1f",
        "avg_city_sanctions_per_100k": ":.2f",
        "n_cities": True,
    },
    color_continuous_scale="YlOrRd",
    mapbox_style="carto-positron",
    zoom=3.2,
    center={"lat": -14.2, "lon": -52.9},
    opacity=0.78,
    title="Brasil por Estado: Sanções por 100 mil com Enriquecimento por Cluster Municipal",
)
fig_state.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_state.show()
        


In [ ]:
# Construir pontos de centróide dos municípios (para mapa das principais cidades)
municipality_zip_path = map_assets_dir / "BR_Municipios_2022.zip"
if not municipality_zip_path.exists():
    municipality_zip_path = download_ibge_boundary_zip("BR_Municipios_2022.zip", output_dir=map_assets_dir)

with tempfile.TemporaryDirectory(prefix="ibge_muni_shape_") as _tmp_dir:
    with zipfile.ZipFile(municipality_zip_path) as _zip_file:
        _zip_file.extractall(_tmp_dir)

    _shp_path = next(Path(_tmp_dir).glob("*.shp"))
    _reader = shapefile.Reader(str(_shp_path))
    _fields = [field[0] for field in _reader.fields[1:]]
    _code_idx = _fields.index("CD_MUN")

    point_rows = []
    for _shape_record in _reader.iterShapeRecords():
        municipality_code = str(_shape_record.record[_code_idx]).zfill(7)
        xmin, ymin, xmax, ymax = _shape_record.shape.bbox
        point_rows.append(
            {
                "municipality_code": municipality_code,
                "lon": (xmin + xmax) / 2.0,
                "lat": (ymin + ymax) / 2.0,
            }
        )

municipality_points = pd.DataFrame(point_rows)
print(f"Pontos de centróide municipal carregados: {len(municipality_points):,}")
municipality_points.head()
        


In [ ]:
# Mapa 2: principais cidades (top por população) com cluster e métricas de compliance
MAIN_CITIES_N = 120

main_cities = (
    city_with_cluster.sort_values("population_2022", ascending=False)
    .head(MAIN_CITIES_N)
    .merge(municipality_points, on="municipality_code", how="left")
    .copy()
)

main_cities["cluster_label"] = main_cities["cluster"].apply(
    lambda x: f"Cluster {int(x)}" if pd.notna(x) else "Sem cluster"
)

missing_points = int(main_cities["lat"].isna().sum())
if missing_points > 0:
    print(f"Aviso: {missing_points} principais cidades sem ponto de centróide; serão excluídas do mapa.")

main_cities_map = main_cities.dropna(subset=["lat", "lon"]).copy()
main_cities_map["population_2022_plot"] = pd.to_numeric(
    main_cities_map["population_2022"], errors="coerce"
).astype(float)

fig_main_cities = px.scatter_mapbox(
    main_cities_map,
    lat="lat",
    lon="lon",
    color="cluster_label",
    size="population_2022_plot",
    size_max=24,
    hover_name="municipality_name",
    hover_data={
        "state_name": True,
        "population_2022": ":,.0f",
        "sanctions_per_100k": ":.2f",
        "avg_income_real_2022_2022_brl": ":.0f",
        "avg_transfer_per_capita": ":.2f",
        "cluster_label": False,
    },
    mapbox_style="carto-positron",
    zoom=3.2,
    center={"lat": -14.2, "lon": -52.9},
    title=f"Principais Cidades do Brasil (Top {MAIN_CITIES_N} por População): Cluster + Métricas de Compliance",
)
fig_main_cities.update_layout(margin={"r": 0, "t": 60, "l": 0, "b": 0})
fig_main_cities.show()

main_cities_map[
    [
        "municipality_name",
        "state_name",
        "population_2022",
        "cluster_label",
        "sanctions_per_100k",
        "avg_income_real_2022_2022_brl",
        "avg_transfer_per_capita",
    ]
].head(20)
        
